In [ ]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizableTextQuery

# 1. Setup
endpoint = "https://rush-lyric-ai-search.search.windows.net"
admin_key = "<REDACTED_API_KEY>"
index_name = "integrated-index"

# 2. Initialize Client
search_client = SearchClient(
    endpoint=endpoint,
    index_name=index_name,
    credential=AzureKeyCredential(admin_key)
)

# 3. Define the Vector Search Query
query_text = "songs about space travel and black holes"

vector_query = VectorizableTextQuery(
    text=query_text, 
    fields="vector", 
    k_nearest_neighbors=3
)

# 4. Execute Search
results = search_client.search(
    search_text=None, 
    vector_queries=[vector_query],
    select=["id", "content"]
)

print(f"Vector Search Results for: '{query_text}'\n")
for result in results:
    score = result.get('@search.score', 0)
    # Print the song ID and a short snippet
    content_snippet = result['content'][:200].replace('\n', ' ')
    print(f"Score: {score:.4f} | ID: {result['id']}")
    print(f"Snippet: {content_snippet}...\n")

Vector Search Results for: 'songs about space travel and black holes'

Score: 0.6568 | ID: cygnus_x1_the_voyage
Snippet: Title: Cygnus X-1: The Voyage Album: A Farewell to Kings Lyrics: Prologue: In the constellation of Cygnus, there lurks a mysterious, invisible force: the black hole of Cygnus X-1.  Six Stars of the No...

Score: 0.6297 | ID: virtuality
Snippet: Title: Virtuality Album: Test for Echo Lyrics: Like a shipwrecked mariner adrift on an unknown sea Clinging to the wreckage of the lost ship Fantasy I'm a castaway, stranded in a desolate land I can s...

Score: 0.6268 | ID: natural_science
Snippet: Title: Natural Science Album: Permanent Waves Lyrics: 1. Tide Pools  When the ebbing tide retreats Along the rocky shoreline It leaves a trail of tidal pools In a short-lived galaxy Each microcosmic p...



In [ ]:
# resetting the index
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexerClient

# 1. Initialize Client
indexer_client = SearchIndexerClient(
    endpoint="https://rush-lyric-ai-search.search.windows.net", 
    credential=AzureKeyCredential("<REDACTED_API_KEY>")
)

# 2. Reset the indexer (Clears its "already processed" list)
indexer_client.reset_indexer("rush-lyrics-indexer")
print("Indexer reset. All documents will be re-processed.")

# 3. Manually trigger the run
indexer_client.run_indexer("rush-lyrics-indexer")
print("Indexer is now running fresh...")

Indexer reset. All documents will be re-processed.
Indexer is now running fresh...


In [ ]:
# Self-contained status check
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexerClient
indexer_client = SearchIndexerClient(
    endpoint="https://rush-lyric-ai-search.search.windows.net", 
    credential=AzureKeyCredential("<REDACTED_API_KEY>")
)
status = indexer_client.get_indexer_status("rush-lyrics-indexer")
last_result = status.last_result
print(f"Indexer Status: {status.status}")
if last_result:
    print(f"Last Run Status: {last_result.status}")
    print(f"Items Processed: {last_result.item_count} / 150")
    print(f"Items Failed: {last_result.failed_item_count}")
    
    if last_result.errors:
        print("\nTOP ERROR:")
        print(last_result.errors[0].error_message)

Indexer Status: running
Last Run Status: success
Items Processed: 150 / 150
Items Failed: 0
